In [1]:
import sys
import os
import warnings
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

from src.config import TRAIN_PATH, TEST_PATH, CV_N_SPLITS, RANDOM_SEED, SUBMISSIONS_DIR
from src.preprocessing import HousePricesPreprocessor
from src.validation import cross_validate
from sklearn.model_selection import KFold


os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [3]:
# Подготовка
y = np.log1p(train['SalePrice'])
test_ids = test['Id']
train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

# Preprocessing
preprocessor = HousePricesPreprocessor()
X = preprocessor.fit_transform(train, np.exp(y) - 1)  # target encoding нужен оригинальный SalePrice
X_test = preprocessor.transform(test)

# Удаление выбросов
outlier_idx = X[(X['GrLivArea'] > 4000) & (np.exp(y) - 1 < 200000)].index
X = X.drop(outlier_idx)
y = y.drop(outlier_idx)

print(f"X: {X.shape}, y: {y.shape}")

X: (1458, 99), y: (1458,)


In [4]:
model = LinearRegression()
scores = cross_validate(model, X, y)

RMSE по фолдам: [0.1075 0.1278 0.1061 0.1186 0.1218]
Среднее: 0.1164 ± 0.0084


In [5]:
# Submission 1: обучение на всех данных (1 фолд)
model_full = LinearRegression()
model_full.fit(X, y)
preds_log = model_full.predict(X_test)
preds = np.expm1(preds_log)  # обратно из log в доллары

sub1 = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
sub1.to_csv(SUBMISSIONS_DIR / 'baseline_1fold.csv', index=False)
print(f"Submission 1 (1 fold): {sub1.shape}")
print(sub1.head())

Submission 1 (1 fold): (1459, 2)
     Id      SalePrice
0  1461  114700.820915
1  1462  151474.278015
2  1463  175140.754333
3  1464  195330.314500
4  1465  197622.784582


In [6]:
# Submission 2: усреднение предсказаний 5 фолдов
kf = KFold(n_splits=CV_N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
test_preds = np.zeros(len(X_test))

for train_idx, val_idx in kf.split(X):
    model_fold = LinearRegression()
    model_fold.fit(X.iloc[train_idx], y.iloc[train_idx])
    test_preds += model_fold.predict(X_test)

test_preds /= CV_N_SPLITS
preds_5fold = np.expm1(test_preds)

sub2 = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_5fold})
sub2.to_csv(SUBMISSIONS_DIR / 'baseline_5fold.csv', index=False)
print(f"\nSubmission 2 (5 fold): {sub2.shape}")
print(sub2.head())


Submission 2 (5 fold): (1459, 2)
     Id      SalePrice
0  1461  114823.533860
1  1462  157650.991347
2  1463  175519.644935
3  1464  195555.235835
4  1465  197685.882583


Итог
- LinearRegression CV (5-fold): 0.1164
- Kaggle 1-fold: 0.14316
- Kaggle 5-fold: 0.14404